# Automata-01 — Automates finis : reconnaître, déterminiser, minimiser

**Navigation** : [Index de la série](README.md) | [Version C#](Automata-01-AutomatesFinis-CSharp.ipynb)

> **Public : Découverte.** Premier notebook de la série `Automata/`. Il ne suppose que
> Python : ensembles, dictionnaires, compréhensions. Tout ce qu'il utilise est construit
> ici, cellule par cellule, puis **vérifié exhaustivement** sur des milliers de mots.

## Ce que ce notebook suppose

- lire une fonction Python avec des `set` et des `dict` ;
- aucune notion d'automate n'est requise : AFD, AFN et minimisation sont définis ici.

## Ce qu'on y apprend

1. **Reconnaître** un mot par un automate fini (déterministe ou non), avec epsilon-transitions.
2. **Déterminiser** par la construction des sous-ensembles, et mesurer que le langage ne change pas.
3. **Minimiser** par l'algorithme de **Hopcroft** (raffinement de partitions), et vérifier
   que deux automates qui reconnaissent le même langage ont le même automate minimal.

**Vérification indépendante** : chaque construction est confrontée à une **référence
externe** (une expression régulière Python `re.fullmatch`, ou une propriété du langage)
sur **tous** les mots jusqu'à une longueur bornée — pas seulement à des exemples choisis.


## 1. Un automate fini déterministe (AFD)

Un AFD est une machine à **états finis** qui lit un mot lettre à lettre, sans mémoire
d'autre chose que son état courant. Formellement : un ensemble d'états $Q$, un alphabet
$\Sigma$, une fonction de transition $\delta : Q \times \Sigma \to Q$, un état
initial $q_0$ et des états **finaux** $F \subseteq Q$. Le mot est **accepté** si la
lecture se termine dans un état final.

Commençons par l'AFD le plus simple qui fasse quelque chose : sur $\Sigma = \{0, 1\}$,
il accepte les mots qui contiennent un **nombre pair de `1`**.


In [1]:
# AFD "nombre pair de 1" : deux etats (pair=0, impair=1), 1 bascule l'etat, 0 le laisse.
# delta[(etat, lettre)] = etat_suivant
delta_pair = {
    (0, '0'): 0, (0, '1'): 1,
    (1, '0'): 1, (1, '1'): 0,
}
ETAT_INITIAL, ETATS_FINAUX = 0, {0}

def afd_accepte(delta, initial, finals, mot):
    """Execute l'AFD sur le mot ; True ssi la lecture finit dans un etat final."""
    q = initial
    for lettre in mot:
        q = delta.get((q, lettre))
        if q is None:          # transition absente = rejet (mot hors alphabet ou blocage)
            return False
    return q in finals

# Quelques mots : le vide a zero '1' (pair) -> accepte ; '101' a deux '1' -> accepte.
for mot in ['', '0', '1', '101', '0110', '111', '1010']:
    print(f"{mot!r:>8} -> {'accepte' if afd_accepte(delta_pair, ETAT_INITIAL, ETATS_FINAUX, mot) else 'rejete'}")


      '' -> accepte
     '0' -> accepte
     '1' -> rejete
   '101' -> accepte
  '0110' -> accepte
   '111' -> rejete
  '1010' -> accepte


**Lecture.** Le mot vide est accepté (zéro est pair) ; `'101'` l'est (deux `1`),
`'111'` ne l'est pas (trois). L'AFD n'a besoin que de **deux** états, parce que la seule
chose à retenir du passé est la **parité** du nombre de `1` — c'est exactement ce que
« mémoire finie » veut dire : tout l'historique du mot se résume à l'état courant.

La **référence externe** de ce langage est immédiate : `mot.count('1') % 2 == 0`.
Vérifions que l'AFD est d'accord avec elle sur **tous** les mots jusqu'à la longueur 10.


In [2]:
from itertools import product

def mots(alphabet, n_max):
    """Tous les mots sur alphabet de longueur 0..n_max ; le mot vide est note ''."""
    out = ['']
    for n in range(1, n_max + 1):
        out.extend(''.join(p) for p in product(alphabet, repeat=n))
    return out

tous = mots(['0', '1'], 10)
divergences = [w for w in tous
               if afd_accepte(delta_pair, ETAT_INITIAL, ETATS_FINAUX, w) != (w.count('1') % 2 == 0)]
print(f"{len(tous)} mots testes (longueur <= 10)")
print(f"Divergences AFD / reference 'parite du nombre de 1' : {len(divergences)}")


2047 mots testes (longueur <= 10)
Divergences AFD / reference 'parite du nombre de 1' : 0


**Lecture.** Zéro divergence sur 2047 mots : l'AFD et la définition en
intension (`count('1') % 2 == 0`) décrivent le **même langage**. C'est la méthode de tout
le notebook : on ne croit une construction qu'après l'avoir confrontée à une référence
**indépendante** sur un domaine fini mais exhaustif.


## 2. Le non-déterminisme et les epsilon-transitions

Un **AFN** (automate fini non déterministe) assouplit la transition :
$\delta : Q \times \Sigma \to \mathcal{P}(Q)$ — une lettre peut mener à
**plusieurs** états, et une **epsilon-transition** ($\varepsilon$) change d'état **sans
lire** de lettre. L'AFN accepte un mot s'il **existe** un chemin réussi.

Le non-déterminisme rend les automates **plus faciles à écrire**. Exemple classique :
les mots sur $\{a, b\}$ qui **finissent par `abb`**. L'AFN « devine » où commence le
`abb` final ; un AFD, lui, doit mémoriser son avancée dans le suffixe à chaque instant.


In [3]:
class AFN:
    """delta : dict (etat, lettre) -> set(etats) ; eps : dict etat -> set(etats) (epsilon)."""
    def __init__(self, etats, alphabet, delta, eps, initial, finals):
        self.etats, self.alphabet = set(etats), set(alphabet)
        self.delta, self.eps = delta, eps
        self.initial, self.finals = initial, set(finals)

    def eclosure(self, S):
        """Ensemble des etats atteignables depuis S par epsilon-transitions seules."""
        S = set(S)
        pile = list(S)
        while pile:
            q = pile.pop()
            for r in self.eps.get(q, set()):
                if r not in S:
                    S.add(r)
                    pile.append(r)
        return frozenset(S)

    def accepte(self, mot):
        """True ssi un chemin de lecture du mot finit dans un etat final."""
        courant = self.eclosure({self.initial})
        for lettre in mot:
            suivant = set()
            for q in courant:
                suivant |= self.delta.get((q, lettre), set())
            courant = self.eclosure(suivant)
            if not courant:
                return False
        return bool(courant & self.finals)

# AFN de (a|b)* abb : etat 0 boucle sur a,b ; on "devine" le a qui ouvre le abb final.
afn_abb = AFN(
    etats={0, 1, 2, 3}, alphabet={'a', 'b'},
    delta={(0, 'a'): {0, 1}, (0, 'b'): {0}, (1, 'b'): {2}, (2, 'b'): {3}},
    eps={}, initial=0, finals={3},
)

for mot in ['abb', 'aabb', 'babb', 'abab', 'abba', 'ab', '']:
    print(f"{mot!r:>6} -> {'accepte' if afn_abb.accepte(mot) else 'rejete'}")


 'abb' -> accepte
'aabb' -> accepte
'babb' -> accepte
'abab' -> rejete
'abba' -> rejete
  'ab' -> rejete
    '' -> rejete


**Lecture.** L'AFN accepte exactement les mots terminés par `abb`
(`'abb'`, `'aabb'`, `'babb'`) et rejette `'abab'`, `'abba'`, `'ab'`. Il n'a que **4**
états, là où l'AFD équivalent en demandera autant mais avec une logique moins lisible.
La référence externe est `mot.endswith('abb')` ; nous la réutiliserons après
déterminisation.


## 3. Déterminiser : la construction des sous-ensembles

Tout AFN a un AFD **équivalent** (même langage) : c'est le théorème de Rabin–Scott.
L'idée : un **état** de l'AFD représente **l'ensemble** des états où l'AFN *pourrait*
être après avoir lu le préfixe courant. On part de l'epsilon-clôture de l'initial, et on
calcule, pour chaque lettre, l'ensemble atteint.

Le prix : un AFN à $n$ états peut demander jusqu'à $2^n$ états déterministes. Ici nous
mesurerons la taille réelle obtenue.


In [4]:
def determiniser(afn):
    """Construction des sous-ensembles. Retourne (etats, delta, initial, finals) de l'AFD ;
    chaque etat de l'AFD est un frozenset d'etats de l'AFN."""
    initial = afn.eclosure({afn.initial})
    etats, delta, finals = {initial}, {}, set()
    a_traiter = [initial]
    while a_traiter:
        S = a_traiter.pop()
        if S & afn.finals:                 # un etat composé est final s'il contient un final
            finals.add(S)
        for lettre in afn.alphabet:
            suivant = set()
            for q in S:
                suivant |= afn.delta.get((q, lettre), set())
            T = afn.eclosure(suivant)
            delta[(S, lettre)] = T
            if T not in etats:
                etats.add(T)
                a_traiter.append(T)
    return etats, delta, initial, finals

def afd_compose_accepte(delta, initial, finals, mot):
    """AFD dont les etats sont des frozensets (issu de determiniser)."""
    q = initial
    for lettre in mot:
        q = delta.get((q, lettre))
        if q is None:
            return False
    return q in finals

etats_abb, delta_abb, init_abb, finals_abb = determiniser(afn_abb)
print(f"AFN : {len(afn_abb.etats)} etats -> AFD : {len(etats_abb)} etats")
print(f"Etats finaux de l'AFD : {len(finals_abb)}")

# Verification exhaustive : AFD == AFN == reference endswith('abb'), sur tous les mots <= 9.
tous_ab = mots(['a', 'b'], 9)
div_afn = [w for w in tous_ab if afn_abb.accepte(w) != afd_compose_accepte(delta_abb, init_abb, finals_abb, w)]
div_ref = [w for w in tous_ab if afd_compose_accepte(delta_abb, init_abb, finals_abb, w) != w.endswith('abb')]
print(f"{len(tous_ab)} mots testes ; divergences AFD/AFN : {len(div_afn)} ; divergences AFD/reference : {len(div_ref)}")


AFN : 4 etats -> AFD : 4 etats
Etats finaux de l'AFD : 1
1023 mots testes ; divergences AFD/AFN : 0 ; divergences AFD/reference : 0


**Lecture.** L'AFN à 4 états se déterminise en un AFD à **4** états
(l'explosion $2^n$ est un borne supérieure, pas une fatalité — ici les sous-ensembles
atteignables sont peu nombreux). Surtout : **zéro divergence** sur 1023 mots, à la fois
contre l'AFN d'origine et contre `endswith('abb')`. La déterminisation préserve le
langage, et le langage est bien celui qu'on visait.

L'AFD obtenu est-il **le plus petit possible** ? C'est la question de la minimisation.


## 4. Minimiser : l'algorithme de Hopcroft

Parmi tous les AFD qui reconnaissent un langage régulier $L$, il en existe un **unique
à isomorphisme près** avec le nombre minimal d'états (théorème de Myhill–Nerode). On
l'obtient en **fusionnant** les états **indiscernables** : deux états $p, q$ sont
discernables s'il existe un mot $w$ accepté depuis l'un et rejeté depuis l'autre.

L'algorithme de **Hopcroft** part de la partition $\{F, Q \setminus F\}$ et la
**raffine** : deux états restent ensemble tant qu'aucune lettre ne les envoie dans des
classes différentes. Complexité $O(n \log n)$ — la meilleure connue pour ce problème.

Pour le voir **réduire** réellement, partons d'un AFD volontairement redondant : les mots
sur $\{a\}$ de **longueur paire**, écrit avec **6** états là où 2 suffisent.


In [5]:
def hopcroft(etats, delta, finals, alphabet):
    """Raffinement de partitions de Hopcroft. Retourne la liste des classes (ensembles
    d'etats indiscernables) de l'AFD minimal. delta a pour cles (etat, lettre)."""
    etats = set(etats)
    P = [b for b in (set(finals), etats - set(finals)) if b]   # partition initiale F / Q\F
    W = [set(b) for b in P]                                     # "splitters" a traiter
    while W:
        A = W.pop()
        for lettre in alphabet:
            X = {q for q in etats if delta.get((q, lettre)) in A}   # preimage de A
            nouvelle = []
            for Y in P:
                inter, diff = Y & X, Y - X
                if inter and diff:                 # Y est coupee en deux par X
                    nouvelle.extend([inter, diff])
                    if Y in W:
                        W.remove(Y)
                        W.extend([set(inter), set(diff)])
                    else:                          # heuristique : on garde la plus petite moitie
                        W.append(set(inter) if len(inter) <= len(diff) else set(diff))
                else:
                    nouvelle.append(Y)
            P = nouvelle
    return P

# AFD "longueur paire" redondant : 6 etats en cycle, finaux = indices pairs.
etats6 = set(range(6))
delta6 = {(q, 'a'): (q + 1) % 6 for q in etats6}
finals6 = {0, 2, 4}
classes = hopcroft(etats6, delta6, finals6, {'a'})
print(f"AFD initial : {len(etats6)} etats ; classes de Hopcroft : {len(classes)}")
for c in sorted(classes, key=sorted):
    print(f"  classe {sorted(c)}")


AFD initial : 6 etats ; classes de Hopcroft : 2
  classe [0, 2, 4]
  classe [1, 3, 5]


**Lecture.** Hopcroft ramène les 6 états à **2 classes** :
$\{0, 2, 4\}$ (positions paires) et $\{1, 3, 5\}$ (impaires). C'est exactement
l'automate minimal du langage « longueur paire » : un aller-retour entre deux états.
L'algorithme a **découvert** la périodicité 2 cachée dans la périodicité 6.

Appliquons-le à notre AFD de `(a|b)*abb` et vérifions une propriété forte : **le minimal
est unique** — deux AFD du même langage doivent donner le même nombre de classes.


In [6]:
# Minimisation de l'AFD de (a|b)*abb (4 etats) : deja minimal ?
classes_abb = hopcroft(etats_abb, delta_abb, finals_abb, {'a', 'b'})
print(f"(a|b)*abb : {len(etats_abb)} etats -> {len(classes_abb)} classes (deja minimal si egal)")

# Propriete d'unicite : l'AFD "pair de 1" (section 1) minimise a 2 classes, comme tout
# AFD correct de ce langage. Verifions sur l'AFD redondant 6 etats du langage "pair".
# Langage de la section 1 = nombre pair de '1' ; reconstruisons-le en 4 etats redondants.
delta4 = {(0,'0'):0,(0,'1'):1,(1,'0'):1,(1,'1'):2,(2,'0'):2,(2,'1'):3,(3,'0'):3,(3,'1'):0}
finals4 = {0, 2}   # deux etats "pair" redondants
classes_pair = hopcroft(set(range(4)), delta4, finals4, {'0','1'})
print(f"parite (version 4 etats) : {len(classes_pair)} classes ; attendu 2 comme la version 2 etats")
assert len(classes_pair) == 2, "l'unicite du minimal est violee"
print("Unicite verifiee : deux AFD du meme langage donnent le meme nombre de classes.")


(a|b)*abb : 4 etats -> 4 classes (deja minimal si egal)
parite (version 4 etats) : 2 classes ; attendu 2 comme la version 2 etats
Unicite verifiee : deux AFD du meme langage donnent le meme nombre de classes.


**Lecture.** `(a|b)*abb` est déjà minimal (4 états → 4 classes) ; le
langage « nombre pair de `1` » minimise à **2** classes, que l'AFD de départ en ait 2 ou 4.
C'est la signature du théorème de Myhill–Nerode : le nombre de classes est une
**invariante du langage**, pas de l'automate choisi.


## 5. Ce que le dépôt fait déjà avec les automates

Ce notebook pose le socle ; d'autres séries **consomment** déjà la notion, chacune sur
son terrain — nous y renvoyons sans les refaire :

- **`Search-10-SymbolicAutomata`** : automates **symboliques**, où les transitions portent
  des prédicats (pas seulement des lettres) — la généralisation directe de nos AFD.
- **`Lean-14-Finiteness-Derivatives`** (+ companion Lean `14b`) : les **dérivées de
  Brzozowski**, une autre route (algébrique) vers la déterminisation et la finitude des
  langages réguliers — preuve formelle à l'appui.
- **`SL-10-ActiveAutomataLearning`** : l'algorithme **L\*** apprend un AFD par requêtes,
  au lieu de le construire à la main.
- **`Z3-04-Strings-Regex`** : le solveur Z3 raisonne sur les **expressions régulières** ;
  c'est le pont vers `Automata-02`, où le théorème de **Kleene** reliera automates et regex.

## Bilan

- Un AFD lit un mot en ne retenant que son **état courant** ; un AFN autorise le
  non-déterminisme et les epsilon-transitions, plus concis à écrire.
- **Déterminiser** (Rabin–Scott) préserve le langage — vérifié exhaustivement ici sur
  1023 mots contre AFN **et** référence externe.
- **Minimiser** (Hopcroft) donne l'AFD **unique** du langage — vérifié par l'invariance
  du nombre de classes sur deux présentations du même langage.


## Exercices

Les trois exercices suivent l'ordre du notebook. Chaque cellule s'exécute **sans erreur
telle qu'elle est livrée** : remplacez les lignes `# TODO etudiant` par votre solution.
La cellule de vérification qui suit chaque énoncé compare votre construction à une
référence externe, comme dans le cours.


### Exercice 1 — Écrire un AFD pour un langage donné

Construisez un AFD sur $\{0, 1\}$ qui accepte les mots se **terminant par `1`**.
Indice : deux états suffisent — « le dernier symbole lu était `1` » ou non. Remplissez
`delta_ex1` et `finals_ex1` (l'état initial est `0`).


In [7]:
# TODO etudiant : completer les transitions et les etats finaux.
delta_ex1 = {
    # (0, '0'): ..., (0, '1'): ...,
    # (1, '0'): ..., (1, '1'): ...,
}
finals_ex1 = set()   # TODO etudiant

# Verification (reference externe : endswith('1')). A executer apres votre construction.
if delta_ex1 and finals_ex1:
    tous = mots(['0', '1'], 9)
    mauvais = [w for w in tous
               if afd_accepte(delta_ex1, 0, finals_ex1, w) != w.endswith('1')]
    print(f"{len(tous)} mots testes ; divergences avec endswith('1') : {len(mauvais)}")
    if mauvais[:5]:
        print("Exemples de desaccord :", mauvais[:5])
else:
    print("Exercice a completer : remplissez delta_ex1 et finals_ex1.")


Exercice a completer : remplissez delta_ex1 et finals_ex1.


### Exercice 2 — Déterminiser un AFN avec epsilon-transitions

Voici un AFN **avec epsilon-transitions** qui reconnaît les mots sur $\{a, b\}$
**finissant par** `aa` **ou par** `b`. Déterminisez-le avec la fonction
`determiniser` du notebook, puis vérifiez contre la référence
`mot.endswith('aa') or mot.endswith('b')`. Combien d'états l'AFD a-t-il ?


In [8]:
afn_ex2 = AFN(
    etats={0, 1, 2, 3},
    alphabet={'a', 'b'},
    delta={(0, 'a'): {0, 1}, (0, 'b'): {0, 3}, (1, 'a'): {2}},
    eps={2: {0}},          # apres avoir vu 'aa', retour gratuit a l'etat 0
    initial=0,
    finals={2, 3},
)

# TODO etudiant : utiliser determiniser(afn_ex2) puis afd_compose_accepte pour verifier.
# etats_e2, delta_e2, init_e2, finals_e2 = determiniser(afn_ex2)
# print(f"AFD : {len(etats_e2)} etats")
# tous = mots(['a', 'b'], 8)
# mauvais = [w for w in tous
#            if afd_compose_accepte(delta_e2, init_e2, finals_e2, w)
#            != (w.endswith('aa') or w.endswith('b'))]
# print(f"Divergences : {len(mauvais)}")
print("Exercice a completer : decommentez le bloc ci-dessus apres avoir lu determiniser().")

Exercice a completer : decommentez le bloc ci-dessus apres avoir lu determiniser().


### Exercice 3 — Montrer qu'un langage n'est pas régulier (contre-exemple de minimisation)

La minimisation suppose un langage **régulier**. Le langage $L = \{a^n b^n \mid n \ge 0\}$
(même nombre de `a` puis de `b`) n'est **pas** régulier : il faudrait « compter » sans
borne, ce qu'une mémoire finie ne peut pas. Ce n'est pas un exercice de code mais de
raisonnement : **expliquez en une phrase**, dans la cellule ci-dessous, pourquoi deux
préfixes $a^i$ et $a^j$ ($i \ne j$) sont **discernables**, et ce que cela implique pour
le nombre d'états d'un hypothétique AFD de $L$.

*(Indice : quel suffixe distingue $a^i$ de $a^j$ ? Pensez à $b^i$.)*


In [9]:
# Reponse a rediger en commentaire (exercice de raisonnement).
# TODO etudiant : deux prefixes a^i et a^j (i != j) sont discernables par le suffixe ...
# ... donc le nombre de classes de Myhill-Nerode est ..., donc un AFD de L aurait ...
reponse_ex3 = None
print("Exercice de raisonnement : redigez votre reponse en commentaire ci-dessus.")


Exercice de raisonnement : redigez votre reponse en commentaire ci-dessus.
